In [1]:


# libraries
import asyncio
import nest_asyncio
import os
from datetime import datetime
from dataclasses import dataclass
import numpy as np
import nea_tools
#from nea_tools import connect, disconnect
import matplotlib.pyplot as plt
import h5py

import ctypes
import sys
import math

# Galvo init


In [2]:
# definitions laser scanner
def galvo_move(Xtarget,Ytarget,CX=0.11080, CY=0.11080):
    gb511_wrap.ctr_goto_xy(int(CX*Xtarget),int(CY*Ytarget))

def galvo_read():
    read=[ctypes.c_long(),ctypes.c_long()]
    #gb511_wrap.ctr_get_current_xy_pos = (read[0], read[1])
    gb511_wrap.ctr_get_current_xy_pos(read[0], read[1])
    return read[0].value,read[1].value

In [3]:
# init galvo

#Load DLL and main galvo functions
gb511_wrap = ctypes.WinDLL(r".\\galvomotor\\CanonGB511.dll")

#When calling functions in the dynamic library, return values and arguments must be of C++ compatible types using "ctypes"
#Set for each command

gb511_wrap.ctr_select_board.argtypes = (ctypes.c_ulong, ctypes.c_ulong)
gb511_wrap.ctr_select_board.restype = ctypes.c_long

gb511_wrap.ctr_load_program_file.argtypes = (ctypes.c_char_p,)
gb511_wrap.ctr_load_program_file.restype = ctypes.c_long

gb511_wrap.ctr_load_correction_file.argtypes = (
    ctypes.c_char_p,
    ctypes.c_double, ctypes.c_double,
    ctypes.c_double,
    ctypes.c_double, ctypes.c_double)
gb511_wrap.ctr_load_correction_file.restype = ctypes.c_long

gb511_wrap.ctr_get_bit_weight.argtypes = (ctypes.c_ulong, ctypes.c_ulong, ctypes.POINTER(ctypes.c_double))
gb511_wrap.ctr_get_bit_weight.restype = ctypes.c_long

gb511_wrap.ctr_set_port_enb.argtypes = (ctypes.c_ulong, ctypes.c_ulong, ctypes.c_ulong, ctypes.c_ulong)
gb511_wrap.ctr_set_port_enb.restype = ctypes.c_long

gb511_wrap.ctr_set_start_list.argtypes = (ctypes.c_ulong, )
gb511_wrap.ctr_set_start_list.restype = ctypes.c_long

gb511_wrap.lst_jump_abs.argtypes = (ctypes.c_long, ctypes.c_long)
gb511_wrap.lst_jump_abs.restype = ctypes.c_long

gb511_wrap.lst_long_delay.argtypes = (ctypes.c_ulong, )
gb511_wrap.lst_long_delay.restype = ctypes.c_long

gb511_wrap.lst_set_end_of_list.argtypes = None
gb511_wrap.lst_set_end_of_list.restype = ctypes.c_long

gb511_wrap.ctr_execute_list.argtypes = (ctypes.c_ulong,)
gb511_wrap.ctr_execute_list.restype = ctypes.c_long

gb511_wrap.ctr_read_status.argtypes = (ctypes.POINTER(ctypes.c_ulong),)
gb511_wrap.ctr_read_status.restype = ctypes.c_long

gb511_wrap.ctr_goto_xy.argtypes = (ctypes.c_long, ctypes.c_long)
gb511_wrap.ctr_goto_xy.restype = ctypes.c_long

gb511_wrap.ctr_get_jump_speed.argtypes = (ctypes.POINTER(ctypes.c_double),)
gb511_wrap.ctr_get_jump_speed.restype = ctypes.c_double

gb511_wrap.ctr_set_jump_speed.argtypes = (ctypes.c_double,)
gb511_wrap.ctr_set_jump_speed.restype = ctypes.c_double

gb511_wrap.ctr_get_current_thermo.argtypes = (ctypes.POINTER(ctypes.c_ulong),)
gb511_wrap.ctr_get_current_thermo.restype = ctypes.c_long

gb511_wrap.ctr_get_current_xy_pos.argtypes = (ctypes.POINTER(ctypes.c_long), ctypes.POINTER(ctypes.c_long))
gb511_wrap.ctr_get_current_xy_pos.restype = ctypes.c_long

gb511_wrap.ctr_get_xy_pos.argtypes = (ctypes.POINTER(ctypes.c_long), ctypes.POINTER(ctypes.c_long))
gb511_wrap.ctr_get_xy_pos.restype = ctypes.c_long

# load dsp file
dspfile = r".\\galvomotor\\gbdsp.hex"
#Filenames must be cp932 encoded
result = gb511_wrap.ctr_load_program_file(dspfile.encode('cp932'))
print('load_dsp_result',result)

# load correction file
#correct_filepath=r".\GM-2020-v2.tsc"
correct_filepath=r".\\galvomotor\\GM-2020-ftheta-10mm-fo4.tsc"
#Filenames must be cp932 encoded
gb511_wrap.ctr_load_correction_file(correct_filepath.encode('cp932'), 1.0, 1.0, 0.0, 0.0, 0.0)

# connect galvo board
nb_board=1; nb_port=2 # 0: PCIe, 1: LAN, 2: USB
result=gb511_wrap.ctr_select_board(nb_port,nb_board)
print('connect_galvo_result',result)

load_dsp_result 0
connect_galvo_result 0


In [6]:
# check galvo
X0,Y0 = galvo_read()
print(X0,Y0)

# init pos
galvo_move(-1912983,225146)
#galvo_move(100000,0) 

print(galvo_read())
print(np.array([galvo_read()])-np.array([-1912983,225146]))

print(galvo_read())

#galvo_move(X0,Y0) 
#print(galvo_read())

100025 7
(15494, 9441)
[[1827287 -204384]]
(-188499, 32268)


In [100]:
# definitions
@dataclass
class ScanResult:
    """Dataclass to combine measured data of mirror scan"""
    o0:np.ndarray
    o1:np.ndarray
    o2:np.ndarray
    o3:np.ndarray
    o4:np.ndarray
    o5:np.ndarray
    coordinates:np.ndarray
    """Measured mirror xyz coordinates in nm"""

    def __iter__(self):
        return iter((self.o0,self.o1,self.o2,self.o3,self.o4,self.o5))

async def scan_mirror(dx, dy, dz, res_x, res_y, res_z, sleep_timer = 0.3):
    """
    Perform a line-by-line scanning movement of the parabolic mirror
    to spatially map the laser intensity in XY plane. Scanning area is a rectangular
    with current mirror motor position as center. Returns measured 
    optical amplitudes as numpy arrays in a dict. Dict keys are indeces of correspondic
    optical harmonic

    Args:
        dx (float): scan range in x axis, in nm
        dy (float): scan range in y axis, in nm
        dz (float): scan range in y axis, in nm
        res_x (int): number of steps in x axis
        res_y (int): number of steps in y axis
        res_z (int): number of steps in z axis
        sleep_timer (float): time to wait after each mirror movement

    Returns:
        Tuple[ScanResult, ScanResult]: Amplitude and phase results with mirror coordinates
    """
    amp: Dict[int, np.ndarray] = {h: np.zeros((res_z, res_y, res_x)) for h in range(6)}
    coords = np.zeros((res_z, res_y, res_x, 3))

    step_x = dx / res_x
    print(step_x)
    step_y = dy / res_y
    print(step_y)
    step_z = dz / res_z
    print(step_z)
    
    #correct for offset in start position
    Xoff=0; Yoff=0; Zoff=0 #nm


    with Mirror() as mirror:
        x0,y0,z0 = mirror.absolute_position
        print(x0, y0, z0)
        mirror.go_relative(-dx/2-Xoff,-dy/2-Yoff,-dz/2-Zoff)
        #mirror.go_relative(-dx/2,-dy/2,-dz/2)
        await mirror.await_async()
        await asyncio.sleep(sleep_timer)

        for iz in range(res_z):
            for iy in range(res_y):
                x_y0, y_y0, z_y0 = mirror.absolute_position
                for ix in range(res_x):
                    x1,y1,z1 = coords[iz,iy,ix] = mirror.absolute_position
                    print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm, z={int(z1-z0)} nm, O2A={round(context.Microscope.Py.OpticalAmplitude[2],3)} mV")
                    
                    for harmonic in range(6):
                        amp[harmonic][iz,iy,ix] = context.Microscope.Py.OpticalAmplitude[harmonic]
                    
                    # stepwise movement in x
                    if ix != res_x-1:
                        mirror.go_relative(step_x, 0, 0) 
                        await mirror.await_async()
                        await asyncio.sleep(sleep_timer)
                    else:
                        x_y1, y_y1, z_y1 = mirror.absolute_position
                        mirror.go_relative(-(x_y1-x_y0), step_y, 0)
                        await mirror.await_async()
                        await asyncio.sleep(sleep_timer)

                    
                    
                # step to next row in y

                #x1,y1,z1 = mirror.absolute_position
                #print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm, z={int(z1-z0)} nm")

            # step to next layer in z
            mirror.go_relative(0,-dy,step_z)
            await mirror.await_async()
            await asyncio.sleep(sleep_timer)
            x1,y1,z1 = mirror.absolute_position
            print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm, z={int(z1-z0)} nm")

        # return to center
        mirror.go_relative(x0-x1,y0-y1,z0-z1)
        await mirror.await_async()
        await asyncio.sleep(sleep_timer)
        xf,yf,zf = mirror.absolute_position
        print(xf, yf, zf)

    amp_result = ScanResult(*(amp[h] for h in range(6)), coords)
    return amp_result

async def scan_galvo_amp(dx, dy, res_x, res_y, tsleep1, tsleep2):
    real_start_time = time.time()
    amp:"dict[int,np.ndarray]" = {}
        
    for harmonic in range(6):
        amp[harmonic] = np.zeros((res_y,res_x))

    coords = np.zeros((res_y,res_x,2)) # position in plane
    
    step_x = dx / (res_x - 1) if res_x > 1 else 0
    step_y = dy / (res_y - 1) if res_y > 1 else 0
    
    # init scan
    x0,y0 = galvo_read()
    galvo_move(x0-dx/2,y0-dy/2)
    #galvo_move(x0-dx/4,y0-dy/4)
    await asyncio.sleep(tsleep2)
    N = res_y * res_x
    for iy in range(res_y):
        for ix in range(res_x): 
            start_time = time.time()

            x_target = x0 - dx/2 + ix * step_x
            y_target = y0 - dy/2 + iy * step_y

            galvo_move(x_target, y_target)
            if ix==0:
                print("NEW LINE")
                tsleep = tsleep2
            else:
                tsleep = tsleep1
            await asyncio.sleep(tsleep)
            x_read, y_read = coords[iy, ix] = galvo_read()
                        
            for harmonic in range(6):
                amp[harmonic][iy,ix] = context.Microscope.Py.OpticalAmplitude[harmonic]
            end_time = time.time()
            print(f"x={int(x_read - x0)} pulses, y={int(y_read - y0)} pulses, O3A={round(context.Microscope.Py.OpticalAmplitude[3],3)} mV, t_tot = {(end_time-start_time):.2f}s, t_meas = {(end_time-start_time-tsleep):.2f}s, {((iy*res_y + ix + 1)*100/N):.2f}%") 
    # return to center
    galvo_move(-1912983,225146) 
    print(galvo_read())
    await asyncio.sleep(tsleep)

    amp_result = ScanResult(amp[0],amp[1],amp[2],amp[3],amp[4],amp[5],coords)
    real_end_time = time.time()
    print("TOTAL TIME:", real_end_time-real_start_time)
    return amp_result



In [112]:
@dataclass
class ScanResult_woO: #No optic measurements here
    coordinates:np.ndarray
        
async def scan(dx, dy, res_x, res_y, tsleep1, tsleep2):
    coords = np.zeros((res_y, res_x, 2))  # galvo coordinates

    step_x = dx/(res_x - 1) if res_x > 1 else 0
    step_y = dy/(res_y - 1) if res_y > 1 else 0

    x0, y0 = galvo_read()
    galvo_move(x0 - dx/2, y0 - dy/2)
    await asyncio.sleep(tsleep2)
    N = res_y * res_x
    for iy in range(res_y):
        start_time_line = time.time()
        for ix in range(res_x):
            start_time = time.time()
            x_target = x0 - dx/2 + ix * step_x
            y_target = y0 - dy/2 + iy * step_y

            galvo_move(x_target, y_target)
            if ix==0:
                tsleep = tsleep2
            elif ix==res_x-1:
                print("NEW LINE")
                time_line = time.time()-start_time_line
                print(f"Time line: {time_line:.2f}s, Time left: {time_line*(res_y-1-iy)}", )
            else:
                tsleep = tsleep1
            await asyncio.sleep(tsleep)

            x_read, y_read = coords[iy, ix] = galvo_read()
            end_time = time.time()
            print(f"x={int(x_read - x0)} pulses, y={int(y_read - y0)} pulses, t_tot = {(end_time-start_time):.2f}s, t_meas = {(end_time-start_time-tsleep):.2f}s, {((iy*res_y + ix + 1)*100/N):.2f}%")

    # Return to center
    galvo_move(-1912983,225146) 
    await asyncio.sleep(tsleep2)
    print(galvo_read())
    scan_result = ScanResult_woO(coords)
    print(scan_result)
    return scan_result

async def immobile(N, tsleep):
    coords = np.zeros((N,2))
    for i in range(N):
        coords[i, :] = galvo_read()
        print(coords[i, :])
        await asyncio.sleep(tsleep)
    
    result = ScanResult_woO(coords)
    return result


In [11]:
# init neasnom
path_to_dll = ""
fingerprint = None
host = 'nea-server'
    
# nea_tools.set_output(None) # turn off logging
    
# connecting and creating module neaspec on success
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)
loop.run_until_complete(nea_tools.connect(host,fingerprint,
                                        path_to_dll))

from neaspec import context
from nea_tools.microscope.motors import Mirror

Trying to download from nea-server
11:29:12.483 Nea.Client.Database Connecting to: Username=neaspec;Database=neaspec;Host=nea-server;Port=5432 
11:29:13.740 Nea.Client.Settings Configuration in local file C:\Users\neaspec\AppData\Local\DefaultDomain_Path_fxmicv33gyl0u4t52k12yjhbwjqhjroe\1.0.0.0\user.config 
11:29:13.777 Nea.Client.Settings Provided by database group: Nea.Client.Hardware.Microscope.Properties.Settings 
11:29:14.986 Nea.Client.Database Cache provides settings of group Nea.Client.Hardware.Camera.Properties.Settings 
11:29:14.986 Nea.Client.Settings Provided by database group: Nea.Client.Hardware.Camera.Properties.Settings 
11:29:15.435 Nea.Client.SDK Waiting for initialization... 
11:29:15.582 Nea.Client.Hardware.Microscope Failed to resolve 10.227.127.150 - No such host is known. System.Net.Sockets.SocketException (0x00002AF9): No such host is known.
   at System.Net.Dns.GetHostEntryOrAddressesCore(IPAddress address, Boolean justAddresses, AddressFamily addressFamily, Nu

In [78]:
# parameters scan VIS
DISTANCE_X = 10000 # scan range in fast axis (x), in nm
DISTANCE_Y = 10000 # scan range in slow axis (y), in nm
DISTANCE_Z = 0 # scan range in slow axis (z), in nm
RESOLUTION_X = 30 # pixels in fast axis (x)
RESOLUTION_Y = 30 # pixels in slow axis (y)
RESOLUTION_Z = 1 # pixels in slow axis (z)
SLEEP_TIMER = 0.2 # time to wait after each movement, in s

In [101]:
# scan
NOW = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
#NOW = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
DIR='C:\\Users\\neaspec\\Documents\\Melaine\\260220 - GalvoScan\\'
#sample_name='pscan-sfg_nftir-tip_785nm-100muW_qcl_5x2s_01'
#FNAME='SiRef_parabola_SNOM-MIR_5000x5000x2500_10x10x3_02s_vis-optz_bis'
#FNAME='SiRef_parabola_SNOM-VIS_4000x4000x1000_10x10x3_02s_galvo-x_2000-y_0'
FNAME=f'BPT_785nm-7mW_parabola_SNOM_{DISTANCE_X}x{DISTANCE_Y}x{DISTANCE_Z}_{RESOLUTION_X}x{RESOLUTION_Y}x{RESOLUTION_Z}'


#DIR= os.path.join(os.environ["HOMEPATH"],"Desktop","ParabolaScan") # directory to save images 
#CMAP = "jet" # color scheme for generated images, see https://matplotlib.org/stable/users/explain/colors/colormaps.html
#FNAME = "ParabolaScan" # beginning of file name of each image

try:
#print('hello')
    if not os.path.exists(DIR):
        os.mkdir(DIR)
    if not os.path.isdir(DIR):
        raise IOError(f"{DIR} is not a directory")
    amp = asyncio.run(scan_mirror(DISTANCE_X,DISTANCE_Y,DISTANCE_Z,RESOLUTION_X,RESOLUTION_Y, RESOLUTION_Z, SLEEP_TIMER))
    
    with h5py.File(os.path.join(DIR,f"{FNAME}_{NOW}.h5"),'w') as file:
        for harmonic, (a) in enumerate(zip(amp)):
            file.create_dataset(f"O{harmonic}",data=a)
        file.create_dataset("coordinates",data=amp.coordinates)

    #for harmonic, array in enumerate(amp):
        #plt.imsave(os.path.join(DIR,f"{FNAME}_O{harmonic}A_{NOW}.png"),array ,cmap=CMAP)

finally: 
    print('Parabola scan done')


333.3333333333333
333.3333333333333
0.0
-8106418.0 -8329303.0 916676.0
x=-5010 nm, y=-4978 nm, z=12 nm, O2A=0.002 mV
x=-4679 nm, y=-4929 nm, z=-20 nm, O2A=0.002 mV
x=-4325 nm, y=-4940 nm, z=3 nm, O2A=0.003 mV
x=-3979 nm, y=-4950 nm, z=1 nm, O2A=0.002 mV
x=-3627 nm, y=-4950 nm, z=5 nm, O2A=0.001 mV
x=-3263 nm, y=-4952 nm, z=4 nm, O2A=0.002 mV
x=-2897 nm, y=-4960 nm, z=7 nm, O2A=0.005 mV
x=-2564 nm, y=-4956 nm, z=3 nm, O2A=0.008 mV
x=-2223 nm, y=-4957 nm, z=12 nm, O2A=0.006 mV
x=-1891 nm, y=-4959 nm, z=8 nm, O2A=0.004 mV
x=-1552 nm, y=-4963 nm, z=3 nm, O2A=0.002 mV
x=-1212 nm, y=-4953 nm, z=3 nm, O2A=0.005 mV
x=-869 nm, y=-4959 nm, z=8 nm, O2A=0.013 mV
x=-530 nm, y=-4959 nm, z=6 nm, O2A=0.018 mV
x=-190 nm, y=-4961 nm, z=8 nm, O2A=0.017 mV
x=137 nm, y=-4964 nm, z=10 nm, O2A=0.029 mV
x=466 nm, y=-4961 nm, z=8 nm, O2A=0.028 mV
x=804 nm, y=-4959 nm, z=9 nm, O2A=0.042 mV
x=1124 nm, y=-4962 nm, z=9 nm, O2A=0.057 mV
x=1467 nm, y=-4964 nm, z=7 nm, O2A=0.072 mV
x=1815 nm, y=-4961 nm, z=12 nm, O2A

x=-4240 nm, y=-3501 nm, z=-12 nm, O2A=0.001 mV
x=-3895 nm, y=-3501 nm, z=-6 nm, O2A=0.002 mV
x=-3563 nm, y=-3501 nm, z=-11 nm, O2A=0.003 mV
x=-3221 nm, y=-3502 nm, z=-2 nm, O2A=0.002 mV
x=-2874 nm, y=-3504 nm, z=-6 nm, O2A=0.003 mV
x=-2532 nm, y=-3507 nm, z=-5 nm, O2A=0.002 mV
x=-2186 nm, y=-3502 nm, z=-2 nm, O2A=0.005 mV
x=-1849 nm, y=-3502 nm, z=-3 nm, O2A=0.009 mV
x=-1511 nm, y=-3501 nm, z=-7 nm, O2A=0.009 mV
x=-1175 nm, y=-3504 nm, z=-8 nm, O2A=0.008 mV
x=-818 nm, y=-3503 nm, z=-2 nm, O2A=0.02 mV
x=-475 nm, y=-3501 nm, z=-8 nm, O2A=0.043 mV
x=-148 nm, y=-3507 nm, z=-10 nm, O2A=0.053 mV
x=184 nm, y=-3507 nm, z=-7 nm, O2A=0.085 mV
x=519 nm, y=-3507 nm, z=-5 nm, O2A=0.123 mV
x=866 nm, y=-3503 nm, z=-5 nm, O2A=0.163 mV
x=1201 nm, y=-3513 nm, z=-2 nm, O2A=0.161 mV
x=1541 nm, y=-3504 nm, z=-1 nm, O2A=0.128 mV
x=1878 nm, y=-3504 nm, z=-5 nm, O2A=0.079 mV
x=2218 nm, y=-3502 nm, z=-4 nm, O2A=0.048 mV
x=2548 nm, y=-3508 nm, z=3 nm, O2A=0.054 mV
x=2882 nm, y=-3509 nm, z=0 nm, O2A=0.067 mV
x=3

x=-3743 nm, y=-2085 nm, z=-12 nm, O2A=0.007 mV
x=-3396 nm, y=-2090 nm, z=-14 nm, O2A=0.012 mV
x=-3034 nm, y=-2086 nm, z=-17 nm, O2A=0.013 mV
x=-2700 nm, y=-2089 nm, z=-14 nm, O2A=0.021 mV
x=-2362 nm, y=-2086 nm, z=-10 nm, O2A=0.028 mV
x=-2028 nm, y=-2092 nm, z=-2 nm, O2A=0.03 mV
x=-1688 nm, y=-2086 nm, z=-12 nm, O2A=0.047 mV
x=-1352 nm, y=-2088 nm, z=-11 nm, O2A=0.064 mV
x=-1003 nm, y=-2089 nm, z=-11 nm, O2A=0.101 mV
x=-669 nm, y=-2089 nm, z=-9 nm, O2A=0.162 mV
x=-353 nm, y=-2092 nm, z=-13 nm, O2A=0.274 mV
x=-17 nm, y=-2092 nm, z=-14 nm, O2A=0.461 mV
x=328 nm, y=-2089 nm, z=-16 nm, O2A=0.618 mV
x=666 nm, y=-2090 nm, z=-15 nm, O2A=0.701 mV
x=1002 nm, y=-2091 nm, z=-14 nm, O2A=0.592 mV
x=1317 nm, y=-2088 nm, z=-17 nm, O2A=0.341 mV
x=1667 nm, y=-2089 nm, z=-10 nm, O2A=0.162 mV
x=2017 nm, y=-2092 nm, z=-10 nm, O2A=0.134 mV
x=2337 nm, y=-2090 nm, z=-7 nm, O2A=0.138 mV
x=2663 nm, y=-2089 nm, z=-11 nm, O2A=0.074 mV
x=3013 nm, y=-2092 nm, z=-14 nm, O2A=0.024 mV
x=3346 nm, y=-2089 nm, z=-9 nm, 

x=-3949 nm, y=-707 nm, z=-19 nm, O2A=0.007 mV
x=-3626 nm, y=-696 nm, z=-23 nm, O2A=0.01 mV
x=-3276 nm, y=-705 nm, z=-15 nm, O2A=0.009 mV
x=-2946 nm, y=-700 nm, z=-24 nm, O2A=0.005 mV
x=-2601 nm, y=-704 nm, z=-20 nm, O2A=0.004 mV
x=-2260 nm, y=-707 nm, z=-22 nm, O2A=0.015 mV
x=-1908 nm, y=-700 nm, z=-16 nm, O2A=0.009 mV
x=-1558 nm, y=-699 nm, z=-16 nm, O2A=0.009 mV
x=-1212 nm, y=-708 nm, z=-14 nm, O2A=0.012 mV
x=-871 nm, y=-704 nm, z=-20 nm, O2A=0.021 mV
x=-522 nm, y=-704 nm, z=-17 nm, O2A=0.347 mV
x=-181 nm, y=-705 nm, z=-19 nm, O2A=1.14 mV
x=155 nm, y=-700 nm, z=-17 nm, O2A=1.826 mV
x=489 nm, y=-704 nm, z=-21 nm, O2A=1.881 mV
x=829 nm, y=-700 nm, z=-18 nm, O2A=1.027 mV
x=1166 nm, y=-705 nm, z=-16 nm, O2A=0.235 mV
x=1509 nm, y=-709 nm, z=-11 nm, O2A=0.17 mV
x=1841 nm, y=-707 nm, z=-17 nm, O2A=0.214 mV
x=2160 nm, y=-709 nm, z=-16 nm, O2A=0.049 mV
x=2495 nm, y=-707 nm, z=-14 nm, O2A=0.045 mV
x=2830 nm, y=-698 nm, z=-17 nm, O2A=0.079 mV
x=3159 nm, y=-709 nm, z=-19 nm, O2A=0.025 mV
x=3500 

x=-2167 nm, y=743 nm, z=-30 nm, O2A=0.06 mV
x=-1820 nm, y=740 nm, z=-21 nm, O2A=0.039 mV
x=-1483 nm, y=740 nm, z=-26 nm, O2A=0.102 mV
x=-1152 nm, y=740 nm, z=-19 nm, O2A=0.404 mV
x=-818 nm, y=740 nm, z=-21 nm, O2A=0.998 mV
x=-492 nm, y=739 nm, z=-24 nm, O2A=2.562 mV
x=-142 nm, y=742 nm, z=-19 nm, O2A=4.407 mV
x=191 nm, y=741 nm, z=-26 nm, O2A=3.46 mV
x=521 nm, y=738 nm, z=-22 nm, O2A=0.62 mV
x=864 nm, y=742 nm, z=-15 nm, O2A=0.063 mV
x=1220 nm, y=740 nm, z=-27 nm, O2A=0.839 mV
x=1563 nm, y=740 nm, z=-21 nm, O2A=0.008 mV
x=1913 nm, y=740 nm, z=-21 nm, O2A=0.029 mV
x=2241 nm, y=742 nm, z=-24 nm, O2A=0.128 mV
x=2569 nm, y=745 nm, z=-18 nm, O2A=0.122 mV
x=2913 nm, y=744 nm, z=-21 nm, O2A=0.044 mV
x=3252 nm, y=741 nm, z=-18 nm, O2A=0.069 mV
x=3590 nm, y=737 nm, z=-18 nm, O2A=0.054 mV
x=3946 nm, y=742 nm, z=-21 nm, O2A=0.009 mV
x=4272 nm, y=740 nm, z=-13 nm, O2A=0.021 mV
x=4606 nm, y=739 nm, z=-18 nm, O2A=0.005 mV
x=4927 nm, y=740 nm, z=-17 nm, O2A=0.005 mV
x=5285 nm, y=740 nm, z=-15 nm, O2A

In [102]:
# for true scans
Xscan =10000# scan range in fast axis (x), in pulse
Yscan = 10000 # scan range in slow axis (y), in pulse
nb_x = 70 # pixels in fast axis (x)
nb_y = 70 # pixels in slow axis (y)
twait1 = 0.025 # time to wait after each movement, in s
twait2 = 0.075

In [107]:
# galvo scan amp
import time
NOW = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

DIR='C:\\Users\\neaspec\\Documents\\Melaine\\260220 - GalvoScan\\'
FNAME=f'NPoM-BPT_785nm-7mW_SNOM_{Xscan}x{Yscan}_{nb_x}x{nb_y}_{twait1}s_{twait2}s'

os.makedirs(DIR, exist_ok=True)
if not os.path.isdir(DIR):
    raise IOError(f"{DIR} is not a directory")

amp = asyncio.run(scan_galvo_amp(Xscan,Yscan,nb_x,nb_y,twait1, twait2))

with h5py.File(os.path.join(DIR,f"{FNAME}_{NOW}.h5"),'w') as file:
    for harmonic, (a) in enumerate(zip(amp)):
        file.create_dataset(f"O{harmonic}",data=a)
    file.create_dataset("coordinates",data=amp.coordinates)

NEW LINE
x=-5391 pulses, y=-4959 pulses, O3A=0.01 mV, t_tot = 0.10s, t_meas = 0.02s, 0.02%
x=-5234 pulses, y=-4950 pulses, O3A=0.01 mV, t_tot = 0.04s, t_meas = 0.02s, 0.04%
x=-5080 pulses, y=-4977 pulses, O3A=0.007 mV, t_tot = 0.06s, t_meas = 0.03s, 0.06%
x=-4945 pulses, y=-4958 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.02s, 0.08%
x=-4802 pulses, y=-4967 pulses, O3A=0.003 mV, t_tot = 0.04s, t_meas = 0.01s, 0.10%
x=-4666 pulses, y=-4952 pulses, O3A=0.01 mV, t_tot = 0.04s, t_meas = 0.02s, 0.12%
x=-4511 pulses, y=-4962 pulses, O3A=0.001 mV, t_tot = 0.04s, t_meas = 0.01s, 0.14%
x=-4360 pulses, y=-4962 pulses, O3A=0.003 mV, t_tot = 0.04s, t_meas = 0.02s, 0.16%
x=-4230 pulses, y=-4950 pulses, O3A=0.005 mV, t_tot = 0.04s, t_meas = 0.01s, 0.18%
x=-4079 pulses, y=-4945 pulses, O3A=0.002 mV, t_tot = 0.05s, t_meas = 0.02s, 0.20%
x=-3936 pulses, y=-4961 pulses, O3A=0.009 mV, t_tot = 0.04s, t_meas = 0.01s, 0.22%
x=-3787 pulses, y=-4959 pulses, O3A=0.005 mV, t_tot = 0.04s, t_meas = 0.02s, 0.24

x=-607 pulses, y=-4819 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.02s, 2.12%
x=-455 pulses, y=-4810 pulses, O3A=0.004 mV, t_tot = 0.03s, t_meas = 0.01s, 2.14%
x=-321 pulses, y=-4809 pulses, O3A=0.005 mV, t_tot = 0.05s, t_meas = 0.02s, 2.16%
x=-154 pulses, y=-4832 pulses, O3A=0.004 mV, t_tot = 0.04s, t_meas = 0.02s, 2.18%
x=-22 pulses, y=-4822 pulses, O3A=0.001 mV, t_tot = 0.05s, t_meas = 0.02s, 2.20%
x=125 pulses, y=-4824 pulses, O3A=0.003 mV, t_tot = 0.04s, t_meas = 0.02s, 2.22%
x=270 pulses, y=-4837 pulses, O3A=0.0 mV, t_tot = 0.04s, t_meas = 0.01s, 2.24%
x=423 pulses, y=-4820 pulses, O3A=0.0 mV, t_tot = 0.04s, t_meas = 0.01s, 2.27%
x=571 pulses, y=-4818 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.02s, 2.29%
x=705 pulses, y=-4828 pulses, O3A=0.004 mV, t_tot = 0.03s, t_meas = 0.01s, 2.31%
x=855 pulses, y=-4811 pulses, O3A=0.004 mV, t_tot = 0.04s, t_meas = 0.02s, 2.33%
x=993 pulses, y=-4824 pulses, O3A=0.004 mV, t_tot = 0.05s, t_meas = 0.03s, 2.35%
x=1143 pulses, y=-4821 pulse

x=4185 pulses, y=-4667 pulses, O3A=0.001 mV, t_tot = 0.05s, t_meas = 0.02s, 4.22%
x=4325 pulses, y=-4681 pulses, O3A=0.001 mV, t_tot = 0.04s, t_meas = 0.01s, 4.24%
x=4474 pulses, y=-4668 pulses, O3A=0.004 mV, t_tot = 0.04s, t_meas = 0.01s, 4.27%
x=4618 pulses, y=-4658 pulses, O3A=0.003 mV, t_tot = 0.04s, t_meas = 0.02s, 4.29%
NEW LINE
x=-5379 pulses, y=-4526 pulses, O3A=0.005 mV, t_tot = 0.10s, t_meas = 0.02s, 4.31%
x=-5238 pulses, y=-4533 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.01s, 4.33%
x=-5081 pulses, y=-4535 pulses, O3A=0.007 mV, t_tot = 0.04s, t_meas = 0.02s, 4.35%
x=-4937 pulses, y=-4516 pulses, O3A=0.008 mV, t_tot = 0.03s, t_meas = 0.01s, 4.37%
x=-4792 pulses, y=-4521 pulses, O3A=0.006 mV, t_tot = 0.04s, t_meas = 0.01s, 4.39%
x=-4661 pulses, y=-4516 pulses, O3A=0.001 mV, t_tot = 0.04s, t_meas = 0.01s, 4.41%
x=-4507 pulses, y=-4526 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.01s, 4.43%
x=-4361 pulses, y=-4530 pulses, O3A=0.005 mV, t_tot = 0.04s, t_meas = 0.02s, 4.45%

x=-1470 pulses, y=-4374 pulses, O3A=0.004 mV, t_tot = 0.04s, t_meas = 0.01s, 6.29%
x=-1327 pulses, y=-4369 pulses, O3A=0.007 mV, t_tot = 0.04s, t_meas = 0.01s, 6.31%
x=-1185 pulses, y=-4384 pulses, O3A=0.008 mV, t_tot = 0.03s, t_meas = 0.01s, 6.33%
x=-1026 pulses, y=-4393 pulses, O3A=0.005 mV, t_tot = 0.04s, t_meas = 0.02s, 6.35%
x=-895 pulses, y=-4397 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.02s, 6.37%
x=-746 pulses, y=-4384 pulses, O3A=0.006 mV, t_tot = 0.05s, t_meas = 0.02s, 6.39%
x=-607 pulses, y=-4377 pulses, O3A=0.005 mV, t_tot = 0.05s, t_meas = 0.02s, 6.41%
x=-450 pulses, y=-4377 pulses, O3A=0.004 mV, t_tot = 0.04s, t_meas = 0.01s, 6.43%
x=-313 pulses, y=-4375 pulses, O3A=0.004 mV, t_tot = 0.04s, t_meas = 0.01s, 6.45%
x=-160 pulses, y=-4368 pulses, O3A=0.001 mV, t_tot = 0.05s, t_meas = 0.02s, 6.47%
x=-13 pulses, y=-4375 pulses, O3A=0.005 mV, t_tot = 0.02s, t_meas = -0.01s, 6.49%
x=133 pulses, y=-4377 pulses, O3A=0.002 mV, t_tot = 0.03s, t_meas = 0.01s, 6.51%
x=268 pulses,

x=3456 pulses, y=-4225 pulses, O3A=0.001 mV, t_tot = 0.04s, t_meas = 0.02s, 8.41%
x=3595 pulses, y=-4240 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.02s, 8.43%
x=3742 pulses, y=-4236 pulses, O3A=0.003 mV, t_tot = 0.04s, t_meas = 0.01s, 8.45%
x=3889 pulses, y=-4248 pulses, O3A=0.007 mV, t_tot = 0.04s, t_meas = 0.02s, 8.47%
x=4029 pulses, y=-4240 pulses, O3A=0.001 mV, t_tot = 0.04s, t_meas = 0.01s, 8.49%
x=4176 pulses, y=-4229 pulses, O3A=0.004 mV, t_tot = 0.04s, t_meas = 0.01s, 8.51%
x=4322 pulses, y=-4237 pulses, O3A=0.002 mV, t_tot = 0.03s, t_meas = 0.01s, 8.53%
x=4472 pulses, y=-4249 pulses, O3A=0.0 mV, t_tot = 0.04s, t_meas = 0.01s, 8.55%
x=4615 pulses, y=-4229 pulses, O3A=0.0 mV, t_tot = 0.04s, t_meas = 0.02s, 8.57%
NEW LINE
x=-5390 pulses, y=-4102 pulses, O3A=0.016 mV, t_tot = 0.09s, t_meas = 0.01s, 8.59%
x=-5238 pulses, y=-4091 pulses, O3A=0.005 mV, t_tot = 0.04s, t_meas = 0.01s, 8.61%
x=-5099 pulses, y=-4090 pulses, O3A=0.014 mV, t_tot = 0.04s, t_meas = 0.02s, 8.63%
x=-4947 

...

x=-4660 pulses, y=4308 pulses, O3A=0.1 mV, t_tot = 0.05s, t_meas = 0.02s, 91.55%
x=-4520 pulses, y=4323 pulses, O3A=0.125 mV, t_tot = 0.05s, t_meas = 0.02s, 91.57%
x=-4364 pulses, y=4315 pulses, O3A=0.124 mV, t_tot = 0.04s, t_meas = 0.01s, 91.59%
x=-4220 pulses, y=4323 pulses, O3A=0.118 mV, t_tot = 0.04s, t_meas = 0.02s, 91.61%
x=-4059 pulses, y=4321 pulses, O3A=0.107 mV, t_tot = 0.04s, t_meas = 0.02s, 91.63%
x=-3936 pulses, y=4299 pulses, O3A=0.098 mV, t_tot = 0.04s, t_meas = 0.02s, 91.65%
x=-3787 pulses, y=4318 pulses, O3A=0.082 mV, t_tot = 0.04s, t_meas = 0.01s, 91.67%
x=-3650 pulses, y=4330 pulses, O3A=0.081 mV, t_tot = 0.04s, t_meas = 0.01s, 91.69%
x=-3499 pulses, y=4314 pulses, O3A=0.074 mV, t_tot = 0.04s, t_meas = 0.01s, 91.71%
x=-3372 pulses, y=4328 pulses, O3A=0.09 mV, t_tot = 0.04s, t_meas = 0.01s, 91.73%
x=-3203 pulses, y=4319 pulses, O3A=0.101 mV, t_tot = 0.05s, t_meas = 0.02s, 91.76%
x=-3069 pulses, y=4296 pulses, O3A=0.116 mV, t_tot = 0.04s, t_meas = 0.02s, 91.78%
x=-2920

x=-314 pulses, y=4470 pulses, O3A=0.078 mV, t_tot = 0.04s, t_meas = 0.01s, 93.59%
x=-168 pulses, y=4464 pulses, O3A=0.095 mV, t_tot = 0.03s, t_meas = 0.01s, 93.61%
x=-14 pulses, y=4467 pulses, O3A=0.091 mV, t_tot = 0.04s, t_meas = 0.01s, 93.63%
x=132 pulses, y=4458 pulses, O3A=0.094 mV, t_tot = 0.04s, t_meas = 0.01s, 93.65%
x=271 pulses, y=4467 pulses, O3A=0.086 mV, t_tot = 0.04s, t_meas = 0.02s, 93.67%
x=410 pulses, y=4467 pulses, O3A=0.078 mV, t_tot = 0.04s, t_meas = 0.02s, 93.69%
x=556 pulses, y=4457 pulses, O3A=0.058 mV, t_tot = 0.04s, t_meas = 0.01s, 93.71%
x=707 pulses, y=4467 pulses, O3A=0.042 mV, t_tot = 0.04s, t_meas = 0.01s, 93.73%
x=838 pulses, y=4459 pulses, O3A=0.041 mV, t_tot = 0.04s, t_meas = 0.01s, 93.76%
x=993 pulses, y=4464 pulses, O3A=0.015 mV, t_tot = 0.05s, t_meas = 0.02s, 93.78%
x=1143 pulses, y=4464 pulses, O3A=0.012 mV, t_tot = 0.04s, t_meas = 0.01s, 93.80%
x=1280 pulses, y=4468 pulses, O3A=0.007 mV, t_tot = 0.04s, t_meas = 0.01s, 93.82%
x=1431 pulses, y=4475 pu

x=4177 pulses, y=4614 pulses, O3A=0.008 mV, t_tot = 0.05s, t_meas = 0.03s, 95.65%
x=4326 pulses, y=4605 pulses, O3A=0.006 mV, t_tot = 0.04s, t_meas = 0.01s, 95.67%
x=4466 pulses, y=4603 pulses, O3A=0.002 mV, t_tot = 0.04s, t_meas = 0.02s, 95.69%
x=4617 pulses, y=4618 pulses, O3A=0.004 mV, t_tot = 0.05s, t_meas = 0.02s, 95.71%
NEW LINE
x=-5387 pulses, y=4752 pulses, O3A=0.082 mV, t_tot = 0.10s, t_meas = 0.02s, 95.73%
x=-5242 pulses, y=4751 pulses, O3A=0.075 mV, t_tot = 0.04s, t_meas = 0.02s, 95.76%
x=-5085 pulses, y=4758 pulses, O3A=0.069 mV, t_tot = 0.04s, t_meas = 0.01s, 95.78%
x=-4944 pulses, y=4741 pulses, O3A=0.066 mV, t_tot = 0.05s, t_meas = 0.02s, 95.80%
x=-4807 pulses, y=4758 pulses, O3A=0.068 mV, t_tot = 0.04s, t_meas = 0.01s, 95.82%
x=-4656 pulses, y=4761 pulses, O3A=0.078 mV, t_tot = 0.03s, t_meas = 0.01s, 95.84%
x=-4519 pulses, y=4757 pulses, O3A=0.1 mV, t_tot = 0.04s, t_meas = 0.02s, 95.86%
x=-4367 pulses, y=4757 pulses, O3A=0.118 mV, t_tot = 0.03s, t_meas = 0.01s, 95.88%
x

x=-1311 pulses, y=4888 pulses, O3A=0.091 mV, t_tot = 0.04s, t_meas = 0.02s, 97.73%
x=-1184 pulses, y=4899 pulses, O3A=0.083 mV, t_tot = 0.04s, t_meas = 0.01s, 97.76%
x=-1041 pulses, y=4894 pulses, O3A=0.073 mV, t_tot = 0.04s, t_meas = 0.01s, 97.78%
x=-891 pulses, y=4901 pulses, O3A=0.065 mV, t_tot = 0.05s, t_meas = 0.02s, 97.80%
x=-751 pulses, y=4905 pulses, O3A=0.063 mV, t_tot = 0.03s, t_meas = 0.01s, 97.82%
x=-612 pulses, y=4907 pulses, O3A=0.05 mV, t_tot = 0.03s, t_meas = 0.01s, 97.84%
x=-451 pulses, y=4901 pulses, O3A=0.044 mV, t_tot = 0.04s, t_meas = 0.01s, 97.86%
x=-313 pulses, y=4911 pulses, O3A=0.047 mV, t_tot = 0.03s, t_meas = 0.01s, 97.88%
x=-158 pulses, y=4898 pulses, O3A=0.049 mV, t_tot = 0.04s, t_meas = 0.01s, 97.90%
x=-22 pulses, y=4890 pulses, O3A=0.066 mV, t_tot = 0.04s, t_meas = 0.02s, 97.92%
x=122 pulses, y=4901 pulses, O3A=0.066 mV, t_tot = 0.04s, t_meas = 0.02s, 97.94%
x=272 pulses, y=4890 pulses, O3A=0.066 mV, t_tot = 0.04s, t_meas = 0.02s, 97.96%
x=415 pulses, y=4

x=3028 pulses, y=5031 pulses, O3A=0.008 mV, t_tot = 0.04s, t_meas = 0.02s, 99.78%
x=3166 pulses, y=5044 pulses, O3A=0.023 mV, t_tot = 0.03s, t_meas = 0.01s, 99.80%
x=3311 pulses, y=5044 pulses, O3A=0.023 mV, t_tot = 0.04s, t_meas = 0.01s, 99.82%
x=3459 pulses, y=5046 pulses, O3A=0.024 mV, t_tot = 0.05s, t_meas = 0.03s, 99.84%
x=3606 pulses, y=5050 pulses, O3A=0.021 mV, t_tot = 0.04s, t_meas = 0.01s, 99.86%
x=3744 pulses, y=5052 pulses, O3A=0.012 mV, t_tot = 0.03s, t_meas = 0.01s, 99.88%
x=3887 pulses, y=5033 pulses, O3A=0.017 mV, t_tot = 0.04s, t_meas = 0.02s, 99.90%
x=4041 pulses, y=5039 pulses, O3A=0.008 mV, t_tot = 0.04s, t_meas = 0.01s, 99.92%
x=4178 pulses, y=5043 pulses, O3A=0.011 mV, t_tot = 0.04s, t_meas = 0.01s, 99.94%
x=4321 pulses, y=5048 pulses, O3A=0.018 mV, t_tot = 0.03s, t_meas = 0.01s, 99.96%
x=4463 pulses, y=5045 pulses, O3A=0.007 mV, t_tot = 0.03s, t_meas = 0.01s, 99.98%
x=4614 pulses, y=5042 pulses, O3A=0.006 mV, t_tot = 0.04s, t_meas = 0.02s, 100.00%
(-1913377, 2251

In [70]:
galvo_move(-1912983,225146)

# for true scans
Xscan =5000# scan range in fast axis (x), in pulse
Yscan = 5000 # scan range in slow axis (y), in pulse
nb_x = 100 # pixels in fast axis (x)
nb_y = 100 # pixels in slow axis (y)
twait1 = 0.02 # time to wait after each movement, in s
twait2 = 0.07

In [71]:
NOW = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

DIR='C:\\Users\\neaspec\\Documents\\Melaine\\260220 - GalvoScan\\'
FNAME=f'scan_{Xscan}x{Yscan}_{nb_x}x{nb_y}_{twait1}s-{twait2}s'

os.makedirs(DIR, exist_ok=True)
if not os.path.isdir(DIR):
    raise IOError(f"{DIR} is not a directory")

res_scan = asyncio.run(scan(Xscan,Yscan,nb_x,nb_y, twait1, twait2))

with h5py.File(os.path.join(DIR,f"{FNAME}_{NOW}.h5"),'w') as file:
    file.create_dataset("coordinates",data=res_scan.coordinates)


x=-2876 pulses, y=-2466 pulses, t_tot = 0.09s, t_meas = 0.02s, 0.01%
x=-2833 pulses, y=-2457 pulses, t_tot = 0.03s, t_meas = 0.01s, 0.02%
x=-2779 pulses, y=-2455 pulses, t_tot = 0.04s, t_meas = 0.02s, 0.03%
x=-2735 pulses, y=-2466 pulses, t_tot = 0.03s, t_meas = 0.01s, 0.04%
x=-2681 pulses, y=-2457 pulses, t_tot = 0.03s, t_meas = 0.01s, 0.05%
x=-2624 pulses, y=-2465 pulses, t_tot = 0.04s, t_meas = 0.02s, 0.06%
x=-2583 pulses, y=-2464 pulses, t_tot = 0.04s, t_meas = 0.02s, 0.07%
x=-2520 pulses, y=-2480 pulses, t_tot = 0.04s, t_meas = 0.02s, 0.08%
x=-2478 pulses, y=-2459 pulses, t_tot = 0.05s, t_meas = 0.03s, 0.09%
x=-2426 pulses, y=-2459 pulses, t_tot = 0.04s, t_meas = 0.02s, 0.10%
x=-2375 pulses, y=-2466 pulses, t_tot = 0.03s, t_meas = 0.01s, 0.11%
x=-2322 pulses, y=-2474 pulses, t_tot = 0.03s, t_meas = 0.01s, 0.12%
x=-2265 pulses, y=-2462 pulses, t_tot = 0.03s, t_meas = 0.01s, 0.13%
x=-2222 pulses, y=-2460 pulses, t_tot = 0.03s, t_meas = 0.01s, 0.14%
x=-2183 pulses, y=-2459 pulses, t_

x=-1817 pulses, y=-2415 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.22%
x=-1767 pulses, y=-2410 pulses, t_tot = 0.04s, t_meas = 0.02s, 1.23%
x=-1728 pulses, y=-2406 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.24%
x=-1664 pulses, y=-2416 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.25%
x=-1616 pulses, y=-2398 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.26%
x=-1568 pulses, y=-2399 pulses, t_tot = 0.04s, t_meas = 0.02s, 1.27%
x=-1508 pulses, y=-2404 pulses, t_tot = 0.04s, t_meas = 0.02s, 1.28%
x=-1472 pulses, y=-2405 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.29%
x=-1412 pulses, y=-2402 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.30%
x=-1372 pulses, y=-2401 pulses, t_tot = 0.04s, t_meas = 0.02s, 1.31%
x=-1321 pulses, y=-2404 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.32%
x=-1266 pulses, y=-2413 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.33%
x=-1212 pulses, y=-2411 pulses, t_tot = 0.03s, t_meas = 0.01s, 1.34%
x=-1160 pulses, y=-2421 pulses, t_tot = 0.04s, t_meas = 0.02s, 1.35%
x=-1118 pulses, y=-2408 pulses, t_

x=-713 pulses, y=-2357 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.44%
x=-664 pulses, y=-2352 pulses, t_tot = 0.04s, t_meas = 0.02s, 2.45%
x=-612 pulses, y=-2353 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.46%
x=-558 pulses, y=-2358 pulses, t_tot = 0.04s, t_meas = 0.02s, 2.47%
x=-504 pulses, y=-2355 pulses, t_tot = 0.04s, t_meas = 0.02s, 2.48%
x=-461 pulses, y=-2346 pulses, t_tot = 0.04s, t_meas = 0.02s, 2.49%
x=-402 pulses, y=-2355 pulses, t_tot = 0.04s, t_meas = 0.02s, 2.50%
x=-371 pulses, y=-2348 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.51%
x=-308 pulses, y=-2361 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.52%
x=-259 pulses, y=-2348 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.53%
x=-215 pulses, y=-2347 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.54%
x=-159 pulses, y=-2347 pulses, t_tot = 0.04s, t_meas = 0.02s, 2.55%
x=-108 pulses, y=-2361 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.56%
x=-62 pulses, y=-2358 pulses, t_tot = 0.03s, t_meas = 0.01s, 2.57%
x=5 pulses, y=-2359 pulses, t_tot = 0.03s, t_meas

x=293 pulses, y=-2309 pulses, t_tot = 0.04s, t_meas = 0.02s, 3.64%
x=355 pulses, y=-2308 pulses, t_tot = 0.03s, t_meas = 0.01s, 3.65%
x=397 pulses, y=-2301 pulses, t_tot = 0.03s, t_meas = 0.01s, 3.66%
x=448 pulses, y=-2312 pulses, t_tot = 0.03s, t_meas = 0.01s, 3.67%
x=504 pulses, y=-2306 pulses, t_tot = 0.03s, t_meas = 0.01s, 3.68%
x=550 pulses, y=-2303 pulses, t_tot = 0.04s, t_meas = 0.02s, 3.69%
x=607 pulses, y=-2304 pulses, t_tot = 0.03s, t_meas = 0.01s, 3.70%
x=657 pulses, y=-2319 pulses, t_tot = 0.04s, t_meas = 0.02s, 3.71%
x=709 pulses, y=-2311 pulses, t_tot = 0.03s, t_meas = 0.01s, 3.72%
x=748 pulses, y=-2304 pulses, t_tot = 0.04s, t_meas = 0.02s, 3.73%
x=813 pulses, y=-2323 pulses, t_tot = 0.03s, t_meas = 0.01s, 3.74%
x=856 pulses, y=-2305 pulses, t_tot = 0.04s, t_meas = 0.02s, 3.75%
x=896 pulses, y=-2305 pulses, t_tot = 0.04s, t_meas = 0.02s, 3.76%
x=956 pulses, y=-2304 pulses, t_tot = 0.05s, t_meas = 0.03s, 3.77%
x=999 pulses, y=-2304 pulses, t_tot = 0.03s, t_meas = 0.01s, 3

x=1415 pulses, y=-2256 pulses, t_tot = 0.05s, t_meas = 0.03s, 4.86%
x=1468 pulses, y=-2263 pulses, t_tot = 0.04s, t_meas = 0.02s, 4.87%
x=1514 pulses, y=-2259 pulses, t_tot = 0.04s, t_meas = 0.02s, 4.88%
x=1562 pulses, y=-2251 pulses, t_tot = 0.04s, t_meas = 0.02s, 4.89%
x=1614 pulses, y=-2247 pulses, t_tot = 0.06s, t_meas = 0.04s, 4.90%
x=1669 pulses, y=-2251 pulses, t_tot = 0.04s, t_meas = 0.02s, 4.91%
x=1720 pulses, y=-2258 pulses, t_tot = 0.06s, t_meas = 0.04s, 4.92%
x=1758 pulses, y=-2261 pulses, t_tot = 0.05s, t_meas = 0.03s, 4.93%
x=1823 pulses, y=-2258 pulses, t_tot = 0.05s, t_meas = 0.03s, 4.94%
x=1871 pulses, y=-2261 pulses, t_tot = 0.04s, t_meas = 0.02s, 4.95%
x=1909 pulses, y=-2258 pulses, t_tot = 0.04s, t_meas = 0.02s, 4.96%
x=1962 pulses, y=-2248 pulses, t_tot = 0.04s, t_meas = 0.02s, 4.97%
x=1998 pulses, y=-2256 pulses, t_tot = 0.05s, t_meas = 0.03s, 4.98%
x=2060 pulses, y=-2253 pulses, t_tot = 0.05s, t_meas = 0.03s, 4.99%
NEW LINE
Time line: 4.60s, Time left: 437.135368

...

x=-2882 pulses, y=2290 pulses, t_tot = 0.09s, t_meas = 0.02s, 94.01%
x=-2837 pulses, y=2285 pulses, t_tot = 0.03s, t_meas = 0.01s, 94.02%
x=-2775 pulses, y=2293 pulses, t_tot = 0.05s, t_meas = 0.03s, 94.03%
x=-2733 pulses, y=2283 pulses, t_tot = 0.05s, t_meas = 0.03s, 94.04%
x=-2683 pulses, y=2295 pulses, t_tot = 0.05s, t_meas = 0.03s, 94.05%
x=-2628 pulses, y=2287 pulses, t_tot = 0.05s, t_meas = 0.03s, 94.06%
x=-2572 pulses, y=2284 pulses, t_tot = 0.06s, t_meas = 0.04s, 94.07%
x=-2521 pulses, y=2273 pulses, t_tot = 0.05s, t_meas = 0.03s, 94.08%
x=-2488 pulses, y=2291 pulses, t_tot = 0.05s, t_meas = 0.03s, 94.09%
x=-2431 pulses, y=2291 pulses, t_tot = 0.03s, t_meas = 0.01s, 94.10%
x=-2375 pulses, y=2291 pulses, t_tot = 0.04s, t_meas = 0.02s, 94.11%
x=-2321 pulses, y=2276 pulses, t_tot = 0.04s, t_meas = 0.02s, 94.12%
x=-2280 pulses, y=2282 pulses, t_tot = 0.04s, t_meas = 0.02s, 94.13%
x=-2229 pulses, y=2276 pulses, t_tot = 0.04s, t_meas = 0.02s, 94.14%
x=-2177 pulses, y=2287 pulses, t_t

x=-1871 pulses, y=2340 pulses, t_tot = 0.05s, t_meas = 0.03s, 95.21%
x=-1815 pulses, y=2335 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.22%
x=-1770 pulses, y=2333 pulses, t_tot = 0.05s, t_meas = 0.03s, 95.23%
x=-1721 pulses, y=2338 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.24%
x=-1671 pulses, y=2336 pulses, t_tot = 0.05s, t_meas = 0.03s, 95.25%
x=-1621 pulses, y=2333 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.26%
x=-1569 pulses, y=2345 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.27%
x=-1504 pulses, y=2344 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.28%
x=-1470 pulses, y=2346 pulses, t_tot = 0.05s, t_meas = 0.03s, 95.29%
x=-1425 pulses, y=2357 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.30%
x=-1374 pulses, y=2340 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.31%
x=-1316 pulses, y=2332 pulses, t_tot = 0.04s, t_meas = 0.02s, 95.32%
x=-1264 pulses, y=2342 pulses, t_tot = 0.03s, t_meas = 0.01s, 95.33%
x=-1214 pulses, y=2343 pulses, t_tot = 0.03s, t_meas = 0.01s, 95.34%
x=-1156 pulses, y=2343 pulses, t_t

x=-750 pulses, y=2391 pulses, t_tot = 0.04s, t_meas = 0.02s, 96.43%
x=-714 pulses, y=2398 pulses, t_tot = 0.04s, t_meas = 0.02s, 96.44%
x=-645 pulses, y=2401 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.45%
x=-630 pulses, y=2390 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.46%
x=-550 pulses, y=2396 pulses, t_tot = 0.04s, t_meas = 0.02s, 96.47%
x=-511 pulses, y=2397 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.48%
x=-460 pulses, y=2394 pulses, t_tot = 0.04s, t_meas = 0.02s, 96.49%
x=-398 pulses, y=2394 pulses, t_tot = 0.04s, t_meas = 0.02s, 96.50%
x=-366 pulses, y=2395 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.51%
x=-317 pulses, y=2402 pulses, t_tot = 0.04s, t_meas = 0.02s, 96.52%
x=-260 pulses, y=2395 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.53%
x=-203 pulses, y=2399 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.54%
x=-145 pulses, y=2399 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.55%
x=-101 pulses, y=2390 pulses, t_tot = 0.03s, t_meas = 0.01s, 96.56%
x=-53 pulses, y=2387 pulses, t_tot = 0.04s, t_me

x=296 pulses, y=2447 pulses, t_tot = 0.04s, t_meas = 0.02s, 97.64%
x=346 pulses, y=2435 pulses, t_tot = 0.05s, t_meas = 0.03s, 97.65%
x=401 pulses, y=2450 pulses, t_tot = 0.04s, t_meas = 0.02s, 97.66%
x=456 pulses, y=2443 pulses, t_tot = 0.04s, t_meas = 0.02s, 97.67%
x=500 pulses, y=2448 pulses, t_tot = 0.04s, t_meas = 0.02s, 97.68%
x=549 pulses, y=2435 pulses, t_tot = 0.03s, t_meas = 0.01s, 97.69%
x=602 pulses, y=2443 pulses, t_tot = 0.04s, t_meas = 0.02s, 97.70%
x=643 pulses, y=2443 pulses, t_tot = 0.04s, t_meas = 0.02s, 97.71%
x=719 pulses, y=2447 pulses, t_tot = 0.03s, t_meas = 0.01s, 97.72%
x=751 pulses, y=2440 pulses, t_tot = 0.03s, t_meas = 0.01s, 97.73%
x=810 pulses, y=2443 pulses, t_tot = 0.04s, t_meas = 0.02s, 97.74%
x=850 pulses, y=2434 pulses, t_tot = 0.05s, t_meas = 0.03s, 97.75%
x=898 pulses, y=2433 pulses, t_tot = 0.05s, t_meas = 0.03s, 97.76%
x=954 pulses, y=2446 pulses, t_tot = 0.03s, t_meas = 0.01s, 97.77%
x=1001 pulses, y=2436 pulses, t_tot = 0.03s, t_meas = 0.01s, 9

x=1306 pulses, y=2496 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.84%
x=1367 pulses, y=2486 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.85%
x=1396 pulses, y=2498 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.86%
x=1459 pulses, y=2500 pulses, t_tot = 0.04s, t_meas = 0.02s, 98.87%
x=1508 pulses, y=2489 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.88%
x=1563 pulses, y=2498 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.89%
x=1615 pulses, y=2482 pulses, t_tot = 0.04s, t_meas = 0.02s, 98.90%
x=1666 pulses, y=2492 pulses, t_tot = 0.04s, t_meas = 0.02s, 98.91%
x=1723 pulses, y=2500 pulses, t_tot = 0.04s, t_meas = 0.02s, 98.92%
x=1760 pulses, y=2499 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.93%
x=1799 pulses, y=2504 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.94%
x=1861 pulses, y=2501 pulses, t_tot = 0.05s, t_meas = 0.03s, 98.95%
x=1915 pulses, y=2500 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.96%
x=1964 pulses, y=2491 pulses, t_tot = 0.03s, t_meas = 0.01s, 98.97%
x=2010 pulses, y=2504 pulses, t_tot = 0.04s, t_m

In [113]:
N=1000
twait=0.05

In [114]:
NOW = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

DIR='C:\\Users\\neaspec\\Documents\\Melaine\\260220 - testGalvo\\'
FNAME=f'immobile_{N}x{twait}'

os.makedirs(DIR, exist_ok=True)
if not os.path.isdir(DIR):
    raise IOError(f"{DIR} is not a directory")

res_scan = asyncio.run(immobile(N,tsleep=twait))

with h5py.File(os.path.join(DIR,f"{FNAME}_{NOW}.h5"),'w') as file:
    file.create_dataset("coordinates",data=res_scan.coordinates)

[-1913375.   225201.]
[-1913356.   225194.]
[-1913376.   225185.]
[-1913367.   225188.]
[-1913358.   225196.]
[-1913358.   225195.]
[-1913356.   225177.]
[-1913374.   225186.]
[-1913367.   225189.]
[-1913379.   225178.]
[-1913363.   225198.]
[-1913364.   225194.]
[-1913375.   225196.]
[-1913363.   225195.]
[-1913358.   225196.]
[-1913379.   225189.]
[-1913376.   225192.]
[-1913365.   225194.]
[-1913373.   225182.]
[-1913373.   225195.]
[-1913371.   225175.]
[-1913372.   225194.]
[-1913372.   225194.]
[-1913369.   225196.]
[-1913363.   225180.]
[-1913365.   225195.]
[-1913358.   225198.]
[-1913376.   225189.]
[-1913362.   225190.]
[-1913369.   225179.]
[-1913368.   225188.]
[-1913368.   225189.]
[-1913366.   225190.]
[-1913358.   225196.]
[-1913357.   225194.]
[-1913362.   225189.]
[-1913370.   225194.]
[-1913350.   225177.]
[-1913360.   225185.]
[-1913359.   225192.]
[-1913359.   225177.]
[-1913358.   225203.]
[-1913360.   225189.]
[-1913368.   225191.]
[-1913347.   225189.]
[-1913363.

[-1913367.   225192.]
[-1913369.   225184.]
[-1913375.   225186.]
[-1913372.   225193.]
[-1913366.   225185.]
[-1913370.   225183.]
[-1913368.   225186.]
[-1913360.   225180.]
[-1913353.   225184.]
[-1913377.   225189.]
[-1913373.   225192.]
[-1913364.   225182.]
[-1913361.   225178.]
[-1913358.   225192.]
[-1913375.   225192.]
[-1913368.   225194.]
[-1913357.   225196.]
[-1913357.   225187.]
[-1913361.   225196.]
[-1913385.   225183.]
[-1913363.   225193.]
[-1913351.   225195.]
[-1913376.   225190.]
[-1913370.   225202.]
[-1913366.   225194.]
[-1913362.   225198.]
[-1913370.   225189.]
[-1913380.   225187.]
[-1913369.   225189.]
[-1913367.   225197.]
[-1913368.   225187.]
[-1913368.   225183.]
[-1913371.   225196.]
[-1913374.   225189.]
[-1913365.   225190.]
[-1913366.   225188.]
[-1913358.   225201.]
[-1913373.   225199.]
[-1913357.   225201.]
[-1913375.   225202.]
[-1913361.   225181.]
[-1913369.   225189.]
[-1913362.   225179.]
[-1913380.   225192.]
[-1913366.   225188.]
[-1913360.

[-1913370.   225196.]
[-1913373.   225188.]
[-1913355.   225190.]
[-1913366.   225180.]
[-1913367.   225185.]
[-1913369.   225180.]
[-1913375.   225189.]
[-1913377.   225197.]
[-1913366.   225198.]
[-1913368.   225191.]
[-1913358.   225196.]
[-1913361.   225199.]
[-1913359.   225183.]
[-1913367.   225193.]
[-1913369.   225192.]
[-1913364.   225184.]
[-1913368.   225182.]
[-1913373.   225191.]
[-1913369.   225192.]
[-1913368.   225179.]
[-1913373.   225187.]
[-1913369.   225180.]
[-1913354.   225192.]
[-1913372.   225185.]
[-1913369.   225189.]
[-1913362.   225203.]
[-1913375.   225209.]
[-1913359.   225197.]
[-1913366.   225199.]
[-1913368.   225193.]
[-1913362.   225171.]
[-1913358.   225188.]
[-1913361.   225190.]
[-1913370.   225189.]
[-1913360.   225178.]
[-1913367.   225191.]
[-1913365.   225197.]
[-1913359.   225184.]
[-1913366.   225191.]
[-1913362.   225185.]
[-1913364.   225188.]
[-1913360.   225191.]
[-1913368.   225195.]
[-1913358.   225184.]
[-1913361.   225178.]
[-1913363.

In [111]:
DIR='C:\\Users\\neaspec\\Documents\\Melaine\\260220 - testGalvo\\'
FNAME=f'immobile_{N}x{twait}'
with h5py.File(os.path.join(DIR,f"{FNAME}_{NOW}.h5"),'w') as file:
    file.create_dataset("coordinates",data=res_scan.coordinates)


ScanResult_woO(coordinates=array([[[-1916242.,   222722.],
        [-1916199.,   222731.],
        [-1916145.,   222733.],
        ...,
        [-1911342.,   222731.],
        [-1911295.,   222710.],
        [-1911241.,   222714.]],

       [[-1916257.,   222784.],
        [-1916203.,   222777.],
        [-1916149.,   222782.],
        ...,
        [-1911354.,   222787.],
        [-1911289.,   222784.],
        [-1911230.,   222772.]],

       [[-1916243.,   222834.],
        [-1916188.,   222843.],
        [-1916146.,   222831.],
        ...,
        [-1911355.,   222832.],
        [-1911307.,   222817.],
        [-1911245.,   222819.]],

       ...,

       [[-1916249.,   227624.],
        [-1916194.,   227635.],
        [-1916136.,   227628.],
        ...,
        [-1911359.,   227620.],
        [-1911296.,   227635.],
        [-1911240.,   227619.]],

       [[-1916245.,   227684.],
        [-1916209.,   227664.],
        [-1916142.,   227680.],
        ...,
        [-1911356.,   2